# Process the PubMed corpus into serialized graphs

This notebook reads the canonical `corpus_articles.parquet`, optionally filters rows by pathogen and publication year, embeds each `node_context.summary`, and saves one NetworkX graph per corpus row under `GRAPH_FOLDER`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib.util
import subprocess
import sys
import warnings
warnings.filterwarnings('ignore', message='IProgress not found.*', module='tqdm.auto')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if importlib.util.find_spec('sentence_transformers') is None:
    print('Installing the optional embedding dependency into the active notebook kernel...')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-e',
        f'{PROJECT_ROOT}[embeddings]',
    ])
from sentence_transformers import SentenceTransformer
from graphicalizer import (
    ExtractionDensityConfig,
    Graphicalizer,
    GraphicalizerConfig,
    NetworkXGraphStore,
    NodeContextConfig,
    NodeEmbeddingConfig,
    load_corpus_articles,
    load_ontology,
    process_corpus_articles,
)

In [ ]:
from graphicalizer.notebook_config import load_notebook_config, resolve_config_path

CONFIG = load_notebook_config(PROJECT_ROOT)
COMMON = CONFIG['common']
SETTINGS = CONFIG['notebook_06_batch_semantic_abstraction']
ASSETS_ROOT = PROJECT_ROOT / 'assets'
CORPUS_PATH = resolve_config_path(
    PROJECT_ROOT,
    Path(COMMON['output_root']) / SETTINGS['corpus_subdir'] / SETTINGS['corpus_filename'],
)
CORPUS_PATHOGENS = SETTINGS['corpus_pathogens']
CORPUS_START_YEAR = SETTINGS['corpus_start_year']
CORPUS_END_YEAR = SETTINGS['corpus_end_year']
GRAPH_FOLDER = resolve_config_path(PROJECT_ROOT, SETTINGS['graph_folder'])
GRAPH_ID_PREFIX = SETTINGS['graph_id_prefix']
CONTINUE_ON_ERROR = SETTINGS['continue_on_error']

ASSEMBLED_ONTOLOGY_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['assembled_ontology_relative_path'])
ONTOLOGY_PATH = (
    ASSEMBLED_ONTOLOGY_PATH
    if ASSEMBLED_ONTOLOGY_PATH.exists()
    else resolve_config_path(PROJECT_ROOT, SETTINGS['base_ontology_relative_path'])
)
PROMPT_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['prompt_relative_path'])
PROMPT_SNAPSHOT_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['prompt_snapshot_relative_path'])
LLM_PROVIDER = SETTINGS['llm_provider']
LLM_MODEL = SETTINGS['openai_model'] if LLM_PROVIDER == 'openai' else SETTINGS['ollama_model']
LLM_OPTIONS = (
    {'max_output_tokens': SETTINGS['openai_max_output_tokens']}
    if LLM_PROVIDER == 'openai'
    else {
        'num_ctx': SETTINGS['ollama_num_ctx'],
        'num_predict': SETTINGS['ollama_num_predict'],
    }
)
EMBEDDING_MODEL_NAME = SETTINGS['embedding_model_name']
EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME)

ontology = load_ontology(ONTOLOGY_PATH)
density = ExtractionDensityConfig(
    entities_per_word=SETTINGS['entities_per_word'],
    relations_per_entity=SETTINGS['relations_per_entity'],
    minimum_entity_fraction=SETTINGS['minimum_entity_fraction'],
    density_retries=SETTINGS['density_retries'],
)
config = GraphicalizerConfig(
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    extraction_density=density,
    node_context=NodeContextConfig(
        max_sentences=SETTINGS['max_context_sentences'],
        max_evidence_items=SETTINGS['max_evidence_items'],
        include_evidence=SETTINGS['include_evidence'],
        include_uncertainty=SETTINGS['include_uncertainty'],
    ),
    prompt_template_path=PROMPT_PATH,
    prompt_snapshot_path=PROMPT_SNAPSHOT_PATH,
    disconnected_policy=SETTINGS['disconnected_policy'],
    context_policy=SETTINGS['context_policy'],
    casting_retries=SETTINGS['casting_retries'],
)
graphicalizer = Graphicalizer.from_provider(
    ontology,
    config,
    options=LLM_OPTIONS,
    embedding_model=EMBEDDING_MODEL,
    embedding_config=NodeEmbeddingConfig(model_id=EMBEDDING_MODEL_NAME),
)
graph_store = NetworkXGraphStore(GRAPH_FOLDER)


In [ ]:
corpus = load_corpus_articles(
    CORPUS_PATH,
    pathogens=CORPUS_PATHOGENS,
    start_year=CORPUS_START_YEAR,
    end_year=CORPUS_END_YEAR,
)
print('Filtered corpus rows:', len(corpus))
batch = process_corpus_articles(
    corpus,
    graphicalizer,
    graph_store,
    corpus_path=CORPUS_PATH,
    graph_id_prefix=GRAPH_ID_PREFIX,
    continue_on_error=CONTINUE_ON_ERROR,
    verbose=True,
)

print('Discovered:', batch.discovered)
print('Processed:', batch.processed)
print('Failed:', batch.failed)
print('Manifest:', batch.manifest_path)
if batch.failures:
    print('Failures:')
    for failure in batch.failures:
        print(failure)